# Fig. S4D: classifier detail, per-batch AUC across models

Per-held-out-batch AUC for sex-adjusted, non-sex, and decorrelated LOBO (pooled AUCs 0.69 / 0.60 / 0.51). Kernel: apa-python.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt

sys.path.insert(0, f"{REPO_ROOT}/figures/style")
import paper_fig_style as S

S.apply_rcparams()
ASD, CTRL = "#d73027", "#4575b4"
DD = f"{DATA_ROOT}/3utr/Blood/2025_analyses/"

In [ ]:
PANEL = "fig_s5d_classifier_detail"
M = {
    "sex-adj (0.69)": "LOBOCV_genelevel_curated_sexadj_ridge",
    "non-sex (0.60)": "LOBOCV_genelevel_curated_ridge",
    "decorrelated (0.51)": "LOBOCV_genelevel_curated_decorrelated_ridge",
}
d = {
    k: pd.read_csv(DD + v + "/per_batch_metrics.csv").set_index("batch")["roc_auc"]
    for k, v in M.items()
}
df = pd.DataFrame(d)
fig, ax = plt.subplots(figsize=(S.FULL_W_PX / 300, S.mm_to_px(52) / 300))
x = np.arange(len(df))
w = 0.26
for j, (k, c) in enumerate(zip(df.columns, ["#2c7fb8", "#80b1d3", "#cccccc"])):
    ax.bar(x + (j - 1) * w, df[k], w, label=k, color=c)
ax.axhline(0.5, ls="--", c="grey", lw=0.7)
ax.set_xticks(x)
ax.set_xticklabels(
    [b.split(":")[0][-4:] + ":" + b.split(":")[1] for b in df.index],
    rotation=30,
    ha="right",
    fontsize=S.FONT_TICK - 1,
)
ax.set_ylabel("held-out-batch AUC", fontsize=S.FONT_LABEL)
ax.set_ylim(0, 1)
ax.legend(fontsize=S.FONT_LEGEND, frameon=False, ncol=3, loc="upper center")
ax.set_title("Per-batch AUC by model (pooled AUCs in legend)", fontsize=S.FONT_ANNOT)
ax.tick_params(labelsize=S.FONT_TICK)
[s.set_visible(False) for s in (ax.spines["top"], ax.spines["right"])]
fig.subplots_adjust(left=0.09, right=0.97, bottom=0.22, top=0.88)
S.save_and_show(fig, PANEL, figure="Fig5/supp_4")

In [ ]:
# pooled confound-sensitivity spectrum
PANEL2 = "fig_s5d_confound_spectrum"
spectrum = pd.DataFrame(
    {
        "model": ["sex-adj", "non-sex\n(primary)", "decorrelated\n(conf-free)"],
        "auc": [0.687, 0.601, 0.505],
        "lo": [0.553, 0.464, 0.363],
        "hi": [0.820, 0.738, 0.648],
    }
)
figs, axs = plt.subplots(figsize=(S.HALF_W_PX / 300, S.mm_to_px(52) / 300))
x = np.arange(len(spectrum))
cols = ["#2c7fb8", "#80b1d3", "#cccccc"]
axs.bar(
    x,
    spectrum.auc,
    color=cols,
    yerr=[spectrum.auc - spectrum.lo, spectrum.hi - spectrum.auc],
    capsize=3,
    error_kw=dict(lw=0.7),
)
axs.axhline(0.5, ls="--", c="grey", lw=0.7)
for i, v in enumerate(spectrum.auc):
    axs.text(i, spectrum.hi[i] + 0.03, f"{v:.2f}", ha="center", fontsize=S.FONT_TICK)
axs.set_xticks(x)
axs.set_xticklabels(spectrum.model, fontsize=S.FONT_TICK)
axs.set_ylim(0, 1)
axs.set_ylabel("pooled LOBO AUC", fontsize=S.FONT_LABEL)
axs.set_title(
    "Confound sensitivity of the classifier\n(collapses to chance when age/sex/composition-\nassociated features are removed)",
    fontsize=S.FONT_ANNOT,
)
axs.tick_params(labelsize=S.FONT_TICK)
[sp.set_visible(False) for sp in (axs.spines["top"], axs.spines["right"])]
figs.subplots_adjust(left=0.16, right=0.96, bottom=0.20, top=0.84)
S.save_and_show(figs, PANEL2, figure="Fig5/supp_4")

### Final-model-only per-batch AUC (forest plot; quarter-height half-width)


In [ ]:
# --- fig_s5d variant: per-batch AUC of the final classifier only ---
# Single model (the final LOBO classifier whose pooled AUC 0.69 is shown in Fig. 4C). No wording
# referring to the alternative (non-sex / decorrelated) models, which are not shown here.
_pb = pd.read_csv(DD + "LOBOCV_genelevel_curated_sexadj_ridge/per_batch_metrics.csv")
_pb["lab"] = [
    b.split(":")[0].replace("VH00", "") + ":" + b.split(":")[1] for b in _pb.batch
]
_pb = _pb.sort_values("roc_auc").reset_index(drop=True)
_POOL = 0.687


def _perbatch_final(figsize, save_name):
    fig, ax = plt.subplots(figsize=figsize)
    y = np.arange(len(_pb))
    for yi, r in zip(y, _pb.itertuples()):
        up = r.roc_auc >= 0.5
        c = "#2c7fb8" if up else "#d73027"
        ax.plot(
            [0.5, r.roc_auc],
            [yi, yi],
            color=c,
            lw=1.4,
            zorder=1,
            solid_capstyle="round",
        )
        ax.scatter(
            r.roc_auc,
            yi,
            s=20 + (r.n_test - 10) * 7,
            color=c,
            zorder=3,
            edgecolors="white",
            linewidths=0.5,
        )
        ax.text(
            r.roc_auc + (0.014 if up else -0.014),
            yi,
            f"{r.roc_auc:.2f}",
            va="center",
            ha="left" if up else "right",
            fontsize=S.FONT_TICK - 0.5,
            color=c,
        )
    ax.axvline(0.5, ls="--", c="grey", lw=0.7, zorder=0)
    ax.axvline(_POOL, ls="-", c="#333333", lw=0.8, zorder=0)
    ax.text(
        _POOL + 0.006,
        0.4,
        f"pooled AUC {_POOL:.2f}",
        rotation=90,
        va="center",
        ha="left",
        fontsize=S.FONT_TICK - 1,
        color="#333",
    )
    ax.set_yticks(y)
    ax.set_yticklabels(
        [f"{r.lab}  (n={r.n_test})" for r in _pb.itertuples()], fontsize=S.FONT_TICK
    )
    ax.set_xlim(0.30, 0.92)
    ax.set_ylim(-0.6, len(_pb) - 0.4)
    ax.set_xlabel("held-out-batch AUC", fontsize=S.FONT_LABEL)
    ax.tick_params(labelsize=S.FONT_TICK, length=0)
    nab = int((_pb.roc_auc >= 0.5).sum())
    ax.set_title(
        "Per-batch performance of the final classifier", fontsize=S.FONT_ANNOT, pad=8
    )
    ax.text(
        0.0,
        1.005,
        f"{nab}/{len(_pb)} batches above chance · dot size ∝ n",
        transform=ax.transAxes,
        fontsize=S.FONT_TICK - 1,
        color="0.45",
        va="bottom",
        ha="left",
    )
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    fig.subplots_adjust(left=0.30, right=0.97, bottom=0.20, top=0.83)
    S.save_and_show(fig, save_name, figure="Fig5/supp_4")


_perbatch_final(
    (S.HALF_W_PX / 300, S.QUARTER_H_PX / 300), "fig_s5d_perbatch_final__quarter_half"
)